In [1]:
!git clone https://github.com/mociatto/AT-SPGD.git

Cloning into 'AT-SPGD'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 357 (delta 10), reused 29 (delta 9), pack-reused 318 (from 1)
Receiving objects: 100% (357/357), 80.60 KiB | 2.18 MiB/s, done.
Resolving deltas: 100% (177/177), done.


In [2]:
%cd AT-SPGD

/kaggle/working/AT-SPGD


In [3]:
!pip install lpips torchmetrics torchattacks

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
  Attempting uninstall: chardet
    Found existing installation: chardet 5.2.0
    Uninstalling chardet-5.2.0:
      Successfully uninstalled charde

In [4]:
from __future__ import annotations

In [5]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [6]:
from typing import Any, Dict, List, Tuple
import time

import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from src.data.datasets import get_dataloaders
from src.engine.evaluator import run_attack_arena
from src.models.split_models import EMB_DIM, ImageClient, VFLServer

DATASETS = ["cifar10", "cifar100", "svhn", "gtsrb"]
MODELS = ["swin_tiny_patch4_window7_224", "resnet18", "mobilenet_v2", "vit_base_patch16_224"]
NUM_SAMPLES = 128
BATCH_SIZE = 128
NUM_WORKERS = 4

WORK_DIR = Path.cwd()
DATA_ROOT = WORK_DIR / "data"
CHECKPOINT_DIR = Path("/kaggle/input/notebooks/mostafaanoosha/spectralvfl/AT-SPGD/checkpoints")
RESULTS_DIR = WORK_DIR / "results"
CSV_DIR = RESULTS_DIR / "csv"
TENSOR_DIR = RESULTS_DIR / "tensors"
OUTPUT_CSV = CSV_DIR / "02_baseline_arena_metrics.csv"


def load_checkpoint(path: Path) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def collect_samples(dataset_name: str) -> Tuple[torch.Tensor, torch.Tensor, int]:
    _, test_loader, num_classes = get_dataloaders(
        dataset_name=dataset_name,
        batch_size=BATCH_SIZE,
        data_root=DATA_ROOT,
        num_workers=NUM_WORKERS,
    )
    image_batches: List[torch.Tensor] = []
    label_batches: List[torch.Tensor] = []

    for images, labels in test_loader:
        image_batches.append(images)
        label_batches.append(labels)
        if sum(batch.size(0) for batch in image_batches) >= NUM_SAMPLES:
            break

    test_batch = torch.cat(image_batches, dim=0)[:NUM_SAMPLES]
    labels = torch.cat(label_batches, dim=0)[:NUM_SAMPLES]
    return test_batch, labels, num_classes

In [7]:
def run_evaluation() -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    TENSOR_DIR.mkdir(parents=True, exist_ok=True)
    total_runs = len(DATASETS) * len(MODELS)
    run_counter = 0
    start_time = time.time()

    dataset_bar = tqdm(DATASETS, desc="Datasets", position=0)
    for dataset_name in dataset_bar:
        dataset_bar.set_postfix(dataset=dataset_name)
        test_batch, labels, dataset_num_classes = collect_samples(dataset_name)

        model_bar = tqdm(MODELS, desc=f"Models ({dataset_name})", position=1, leave=False)
        for model_name in model_bar:
            run_counter += 1
            checkpoint_path = CHECKPOINT_DIR / f"01_baseline_{dataset_name}_{model_name}.pth"
            elapsed_min = (time.time() - start_time) / 60.0
            model_bar.set_postfix(
                model=model_name,
                run=f"{run_counter}/{total_runs}",
                elapsed_min=f"{elapsed_min:.1f}",
            )
            tqdm.write(
                f"[Attack Arena] Dataset={dataset_name} | Model={model_name} | "
                f"Run={run_counter}/{total_runs} | Checkpoint={checkpoint_path.name}"
            )
            checkpoint = load_checkpoint(checkpoint_path)
            num_classes = int(checkpoint.get("num_classes", dataset_num_classes))

            client = ImageClient(model_name=model_name, dim=EMB_DIM)
            server = VFLServer(emb_dim=EMB_DIM, num_classes=num_classes)
            client.load_state_dict(checkpoint["image_client"])
            server.load_state_dict(checkpoint["vfl_server"])
            client.eval()
            server.eval()

            attack_rows, artifact_tensors = run_attack_arena(client, server, test_batch, labels)
            artifact_path = TENSOR_DIR / f"02_artifacts_{dataset_name}_{model_name}.pt"
            torch.save(artifact_tensors, artifact_path)
            for row in attack_rows:
                rows.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "num_classes": num_classes,
                        "artifact_path": str(artifact_path),
                        **row,
                    }
                )

            del client, server, artifact_tensors
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        model_bar.close()

        del test_batch, labels
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dataset_bar.close()
    return pd.DataFrame(rows)


arena_df = run_evaluation()

Datasets:   0%|          | 0/4 [00:00<?, ?it/s]


  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 98.3k/170M [00:00<04:32, 625kB/s]
  0%|          | 197k/170M [00:00<03:56, 721kB/s] 
  0%|          | 295k/170M [00:00<03:57, 715kB/s]
  0%|          | 393k/170M [00:00<04:15, 665kB/s]
  0%|          | 492k/170M [00:00<04:10, 679kB/s]
  0%|          | 590k/170M [00:00<04:24, 642kB/s]
  0%|          | 688k/170M [00:01<04:06, 688kB/s]
  0%|          | 786k/170M [00:01<04:04, 693kB/s]
  1%|          | 885k/170M [00:01<04:03, 696kB/s]
  1%|          | 983k/170M [00:01<04:01, 702kB/s]
  1%|          | 1.08M/170M [00:01<04:04, 693kB/s]
  1%|          | 1.18M/170M [00:01<04:04, 693kB/s]
  1%|          | 1.28M/170M [00:01<04:01, 700kB/s]
  1%|          | 1.38M/170M [00:01<04:04, 693kB/s]
  1%|          | 1.47M/170M [00:02<04:01, 699kB/s]
  1%|          | 1.57M/170M [00:02<04:05, 689kB/s]
  1%|          | 1.67M/170M [00:02<04:01, 698kB/s]
  1%|          | 1.77M/170M [00:02<04:09, 677kB/s]
  1%|          | 1.87M/170M [00:02<04:01, 698k

Models (cifar10):   0%|          | 0/4 [00:00<?, ?it/s]

[Attack Arena] Dataset=cifar10 | Model=swin_tiny_patch4_window7_224 | Run=1/16 | Checkpoint=01_baseline_cifar10_swin_tiny_patch4_window7_224.pth


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

...

Models (svhn):   0%|          | 0/4 [00:00<?, ?it/s]

[Attack Arena] Dataset=svhn | Model=swin_tiny_patch4_window7_224 | Run=9/16 | Checkpoint=01_baseline_svhn_swin_tiny_patch4_window7_224.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=svhn | Model=resnet18 | Run=10/16 | Checkpoint=01_baseline_svhn_resnet18.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=svhn | Model=mobilenet_v2 | Run=11/16 | Checkpoint=01_baseline_svhn_mobilenet_v2.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=svhn | Model=vit_base_patch16_224 | Run=12/16 | Checkpoint=01_baseline_svhn_vit_base_patch16_224.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1]


  0%|          | 0.00/187M [00:00<?, ?B/s]
  0%|          | 32.8k/187M [00:00<22:43, 137kB/s]
  0%|          | 98.3k/187M [00:00<10:10, 307kB/s]
  0%|          | 164k/187M [00:00<07:55, 394kB/s] 
  0%|          | 360k/187M [00:00<03:48, 821kB/s]
  0%|          | 688k/187M [00:00<02:09, 1.45MB/s]
  1%|          | 1.41M/187M [00:00<01:04, 2.90MB/s]
  2%|▏         | 2.82M/187M [00:00<00:32, 5.64MB/s]
  3%|▎         | 5.67M/187M [00:01<00:16, 11.2MB/s]
  4%|▍         | 7.21M/187M [00:01<00:15, 11.5MB/s]
  5%|▍         | 8.85M/187M [00:01<00:14, 12.2MB/s]
  6%|▌         | 11.0M/187M [00:01<00:12, 13.8MB/s]
  8%|▊         | 14.9M/187M [00:01<00:08, 19.4MB/s]
  9%|▉         | 17.5M/187M [00:01<00:08, 20.0MB/s]
 11%|█         | 21.0M/187M [00:01<00:07, 22.8MB/s]
 13%|█▎        | 24.0M/187M [00:01<00:07, 23.2MB/s]
 15%|█▍        | 27.4M/187M [00:02<00:06, 24.7MB/s]
 16%|█▌        | 30.3M/187M [00:02<00:06, 24.6MB/s]
 18%|█▊        | 33.5M/187M [00:02<00:06, 24.9MB/s]
 19%|█▉        | 36.4M/187

Models (gtsrb):   0%|          | 0/4 [00:00<?, ?it/s]

[Attack Arena] Dataset=gtsrb | Model=swin_tiny_patch4_window7_224 | Run=13/16 | Checkpoint=01_baseline_gtsrb_swin_tiny_patch4_window7_224.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=gtsrb | Model=resnet18 | Run=14/16 | Checkpoint=01_baseline_gtsrb_resnet18.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=gtsrb | Model=mobilenet_v2 | Run=15/16 | Checkpoint=01_baseline_gtsrb_mobilenet_v2.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[Attack Arena] Dataset=gtsrb | Model=vit_base_patch16_224 | Run=16/16 | Checkpoint=01_baseline_gtsrb_vit_base_patch16_224.pth
Setting up [LPIPS] perceptual loss: trunk [vgg

In [8]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
arena_df = arena_df.sort_values(["dataset", "model", "attack"]).reset_index(drop=True)
arena_df.to_csv(OUTPUT_CSV, index=False)
display(arena_df)

,dataset,model,num_classes,artifact_path,attack,eps,alpha,steps,clean_accuracy,asr,psnr,ssim,lpips
0,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,APGD,0.031373,0.007843,10,0.898438,1.0,34.117175,0.806010,0.446258
1,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,AT-SPGD,0.031373,0.007843,10,0.898438,1.0,42.392912,0.965589,0.320752
2,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,MIFGSM,0.031373,0.007843,10,0.898438,1.0,31.215459,0.688170,0.538241
3,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,PGD,0.031373,0.007843,10,0.898438,1.0,33.953114,0.795648,0.482033
4,cifar10,mobilenet_v2,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,SSA,0.031373,0.007843,10,0.898438,1.0,31.299755,0.689220,0.548645
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,svhn,vit_base_patch16_224,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,APGD,0.031373,0.007843,10,0.640625,1.0,35.556796,0.837398,0.365075
76,svhn,vit_base_patch16_224,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,AT-SPGD,0.031373,0.007843,10,0.640625,1.0,42.047827,0.955085,0.373135
77,svhn,vit_base_patch16_224,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,MIFGSM,0.031373,0.007843,10,0.640625,1.0,31.121807,0.637557,0.654485
78,svhn,vit_base_patch16_224,10,/kaggle/working/AT-SPGD/results/tensors/02_art...,PGD,0.031373,0.007843,10,0.640625,1.0,33.876142,0.754359,0.555308
